In [1]:
%pip install "gymnasium[box2d]" stable-baselines3[extra] torch opencv-python pyyaml moviepy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os
import json
import numpy as np
import gymnasium as gym
import cv2

from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecMonitor, VecTransposeImage, VecVideoRecorder


In [ ]:
# ========= CONFIGURACIÓN =========
RUN_DIR = Path("experiments/carracing_ppo/run_20260423_171321_seed2")

MODEL_PATH = RUN_DIR / "best_model" / "best_model.zip"

ENV_ID = "CarRacing-v3"
SEED = 0

CONTINUOUS = True
GRAYSCALE = True
N_STACK = 4

N_EVAL_EPISODES = 20
DETERMINISTIC = True

LAP_COMPLETE_PERCENT = 0.95

VIDEO_DIR = RUN_DIR / "videos"
VIDEO_PREFIX = "best_model_eval"
VIDEO_EPISODE_LENGTH = 2000   # duración máxima por video
N_VIDEO_EPISODES = 3

assert RUN_DIR.exists(), f"No existe RUN_DIR: {RUN_DIR}"
assert MODEL_PATH.exists(), f"No existe MODEL_PATH: {MODEL_PATH}"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_DIR:", RUN_DIR.resolve())
print("MODEL_PATH:", MODEL_PATH.resolve())
print("VIDEO_DIR:", VIDEO_DIR.resolve())


RUN_DIR: C:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\09_validation_multiseed_seed2\experiments\carracing_ppo\run_20260423_171321_seed2
MODEL_PATH: C:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\09_validation_multiseed_seed2\experiments\carracing_ppo\run_20260423_171321_seed2\best_model\best_model.zip
VIDEO_DIR: C:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\09_validation_multiseed_seed2\experiments\carracing_ppo\run_20260423_171321_seed2\videos


In [4]:
# ========= WRAPPERS =========

class GrayscaleObservation(gym.ObservationWrapper):
    """Convierte RGB uint8 (H, W, 3) a grayscale uint8 (H, W, 1)."""

    def __init__(self, env: gym.Env):
        super().__init__(env)
        assert isinstance(env.observation_space, spaces.Box)
        obs_space = env.observation_space
        assert len(obs_space.shape) == 3 and obs_space.shape[-1] == 3, (
            f"Expected channel-last RGB image, got shape={obs_space.shape}"
        )
        h, w, _ = obs_space.shape
        self.observation_space = spaces.Box(
            low=0,
            high=255,
            shape=(h, w, 1),
            dtype=np.uint8,
        )

    def observation(self, obs: np.ndarray) -> np.ndarray:
        gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        return gray[..., None].astype(np.uint8)


In [ ]:
# ========= FACTORY DEL ENTORNO =========

def build_single_env(seed: int, monitor_file: Path | None = None, render_mode: str | None = None):
    env = gym.make(
        ENV_ID,
        continuous=CONTINUOUS,
        render_mode=render_mode,
        lap_complete_percent=LAP_COMPLETE_PERCENT,
    )

    env.action_space.seed(seed)
    env.reset(seed=seed)

    if GRAYSCALE:
        env = GrayscaleObservation(env)

    if monitor_file is not None:
        env = Monitor(env, filename=str(monitor_file))
    else:
        env = Monitor(env)

    return env


def make_eval_vec_env(seed: int = 0, with_video: bool = False):
    render_mode = "rgb_array" if with_video else None

    def _init():
        return build_single_env(
            seed=seed,
            monitor_file=RUN_DIR / f"monitor_eval_notebook_seed{seed}.csv",
            render_mode=render_mode,
        )

    vec_env = DummyVecEnv([_init])
    vec_env = VecMonitor(vec_env, filename=str(RUN_DIR / "vec_monitor_eval_notebook.csv"))
    vec_env = VecFrameStack(vec_env, n_stack=N_STACK)
    vec_env = VecTransposeImage(vec_env)

    return vec_env


def unwrap_to_base_env(vec_env):
    current = vec_env
    while hasattr(current, "venv"):
        current = current.venv
    if hasattr(current, "envs") and len(current.envs) > 0:
        return current.envs[0].unwrapped
    raise RuntimeError("No se pudo acceder al entorno base.")


def episode_completion_status(vec_env):
    base_env = unwrap_to_base_env(vec_env)
    track = getattr(base_env, "track", None)
    tile_visited_count = getattr(base_env, "tile_visited_count", None)
    lap_complete_percent = getattr(base_env, "lap_complete_percent", LAP_COMPLETE_PERCENT)

    if track is None or tile_visited_count is None or len(track) == 0:
        return False

    visited_fraction = tile_visited_count / len(track)
    return visited_fraction >= lap_complete_percent


In [6]:
# ========= CARGA DEL MODELO =========

eval_env = make_eval_vec_env(seed=SEED, with_video=False)

print("Observation space:", eval_env.observation_space)
print("Action space:", eval_env.action_space)

model = PPO.load(MODEL_PATH, env=eval_env)

print("Modelo cargado correctamente.")


c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\.venv\Lib\site-packages\stable_baselines3\common\vec_env\vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Observation space: Box(0, 255, (4, 96, 96), uint8)
Action space: Box([-1.  0.  0.], 1.0, (3,), float32)
Modelo cargado correctamente.


In [7]:
# ========= EVALUACIÓN CUANTITATIVA =========

mean_reward, std_reward = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=N_EVAL_EPISODES,
    deterministic=DETERMINISTIC,
    render=False,
    return_episode_rewards=False,
)

print(f"Mean reward over {N_EVAL_EPISODES} eval episodes: {mean_reward:.3f}")
print(f"Std reward over  {N_EVAL_EPISODES} eval episodes: {std_reward:.3f}")
print("\nNota: el completion rate se calcula en la celda 'EVALUACIÓN EPISODIO POR EPISODIO'.")


Mean reward over 20 eval episodes: 718.719
Std reward over  20 eval episodes: 195.165

Nota: el completion rate se calcula en la celda 'EVALUACIÓN EPISODIO POR EPISODIO'.


In [8]:
# ========= EVALUACIÓN EPISODIO POR EPISODIO =========

import pandas as pd

episode_rewards = []
episode_lengths = []
episode_completed = []
records = []

for ep in range(N_EVAL_EPISODES):
    obs = eval_env.reset()
    total_reward = 0.0
    steps = 0

    while True:
        action, _states = model.predict(obs, deterministic=DETERMINISTIC)
        obs, rewards, dones, infos = eval_env.step(action)

        total_reward += float(rewards[0])
        steps += 1

        if dones[0]:
            break

    completed = episode_completion_status(eval_env)

    episode_rewards.append(total_reward)
    episode_lengths.append(steps)
    episode_completed.append(bool(completed))
    records.append({
        "episode": ep + 1,
        "reward": total_reward,
        "length": steps,
        "completed_lap": bool(completed),
    })

    print(
        f"Episode {ep+1}: reward={total_reward:.3f}, length={steps}, completed_lap={completed}"
    )

results_df = pd.DataFrame(records)
completion_rate = 100.0 * np.mean(episode_completed) if episode_completed else 0.0

print("\nResumen:")
print(f"Reward medio:      {np.mean(episode_rewards):.3f}")
print(f"Reward std:        {np.std(episode_rewards):.3f}")
print(f"Length medio:      {np.mean(episode_lengths):.3f}")
print(f"Completion rate:   {completion_rate:.1f}% ({sum(episode_completed)}/{len(episode_completed)})")

results_df


Episode 1: reward=799.254, length=1000, completed_lap=False
Episode 2: reward=375.083, length=1000, completed_lap=False
Episode 3: reward=654.777, length=1000, completed_lap=False
Episode 4: reward=878.723, length=1000, completed_lap=False
Episode 5: reward=835.484, length=1000, completed_lap=False
Episode 6: reward=795.307, length=1000, completed_lap=False
Episode 7: reward=73.333, length=1000, completed_lap=False
Episode 8: reward=695.107, length=1000, completed_lap=False
Episode 9: reward=15.463, length=484, completed_lap=False
Episode 10: reward=660.000, length=1000, completed_lap=False
Episode 11: reward=858.333, length=1000, completed_lap=False
Episode 12: reward=924.300, length=757, completed_lap=False
Episode 13: reward=900.800, length=992, completed_lap=False
Episode 14: reward=865.385, length=1000, completed_lap=False
Episode 15: reward=184.507, length=1000, completed_lap=False
Episode 16: reward=328.571, length=1000, completed_lap=False
Episode 17: reward=864.539, length=100

,episode,reward,length,completed_lap
0,1,799.253750,1000,False
1,2,375.083043,1000,False
2,3,654.777088,1000,False
3,4,878.723400,1000,False
4,5,835.483885,1000,False
5,6,795.306840,1000,False
6,7,73.333333,1000,False
7,8,695.107012,1000,False
8,9,15.463063,484,False
9,10,660.000002,1000,False


In [ ]:
# ========= GRABAR VIDEOS =========

video_env = make_eval_vec_env(seed=SEED + 123, with_video=True)

video_env = VecVideoRecorder(
    video_env,
    video_folder=str(VIDEO_DIR),
    record_video_trigger=lambda step: step == 0,
    video_length=VIDEO_EPISODE_LENGTH,
    name_prefix=VIDEO_PREFIX,
)

video_model = PPO.load(MODEL_PATH, env=video_env)

for ep in range(N_VIDEO_EPISODES):
    obs = video_env.reset()
    total_reward = 0.0
    steps = 0

    while True:
        action, _ = video_model.predict(obs, deterministic=DETERMINISTIC)
        obs, rewards, dones, infos = video_env.step(action)

        total_reward += float(rewards[0])
        steps += 1

        if dones[0] or steps >= VIDEO_EPISODE_LENGTH:
            break

    print(f"Video episode {ep+1}: reward={total_reward:.3f}, length={steps}")

video_env.close()

print("\nVideos generados en:")
print(VIDEO_DIR.resolve())
print("\nArchivos:")
for p in sorted(VIDEO_DIR.glob("*.mp4")):
    print(" -", p.name)


c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\.venv\Lib\site-packages\stable_baselines3\common\vec_env\vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Video episode 1: reward=261.194, length=1000
Saving video to c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\09_validation_multiseed_seed2\experiments\carracing_ppo\run_20260423_171321_seed2\videos\best_model_eval-step-0-to-step-2000.mp4
MoviePy - Building video c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\09_validation_multiseed_seed2\experiments\carracing_ppo\run_20260423_171321_seed2\videos\best_model_eval-step-0-to-step-2000.mp4.
MoviePy - Writing video c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\09_validation_multiseed_seed2\experiments\carracing_ppo\run_20260423_171321_seed2\videos\best_model_eval-step-0-to-step-2000.mp4



MoviePy - Done !
MoviePy - video ready c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\09_validation_multiseed_seed2\experiments\carracing_ppo\run_20260423_171321_seed2\videos\best_model_eval-step-0-to-step-2000.mp4
Video episode 2: reward=599.324, length=1000
Video episode 3: reward=390.446, length=1000

Videos generados en:
C:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\09_validation_multiseed_seed2\experiments\carracing_ppo\run_20260423_171321_seed2\videos

Archivos:
 - best_model_eval-step-0-to-step-2000.mp4


In [ ]:
# ========= VISUALIZAR CARPETA DE LA CORRIDA =========

print("Contenido principal de RUN_DIR:")
for p in sorted(RUN_DIR.iterdir()):
    print(" -", p.name)

print("\nVideos encontrados:")
for p in sorted(VIDEO_DIR.glob("*")):
    print(" -", p.name)


Contenido principal de RUN_DIR:
 - best_model
 - checkpoints
 - config.json
 - config.yaml
 - eval
 - final_model.zip
 - monitor_eval_10000.csv.monitor.csv
 - monitor_eval_notebook_seed0.csv.monitor.csv
 - monitor_eval_notebook_seed123.csv.monitor.csv
 - monitor_train_0.csv.monitor.csv
 - monitor_train_1.csv.monitor.csv
 - monitor_train_2.csv.monitor.csv
 - monitor_train_3.csv.monitor.csv
 - monitor_train_4.csv.monitor.csv
 - monitor_train_5.csv.monitor.csv
 - monitor_train_6.csv.monitor.csv
 - monitor_train_7.csv.monitor.csv
 - tb
 - vec_monitor_eval.csv.monitor.csv
 - vec_monitor_eval_notebook.csv.monitor.csv
 - vec_monitor_train.csv.monitor.csv
 - videos

Videos encontrados:
 - best_model_eval-step-0-to-step-2000.mp4


In [11]:
import numpy as np
import pandas as pd

evaluation_path = RUN_DIR / "eval" / "evaluations.npz"
data = np.load(evaluation_path)

timesteps = data["timesteps"]
results = data["results"]
ep_lengths = data["ep_lengths"]

mean_rewards = results.mean(axis=1)
std_rewards = results.std(axis=1)
min_rewards = results.min(axis=1)
max_rewards = results.max(axis=1)

mean_lengths = ep_lengths.mean(axis=1)
std_lengths = ep_lengths.std(axis=1)

df = pd.DataFrame({
    "timesteps": timesteps,
    "reward_mean": mean_rewards,
    "reward_std": std_rewards,
    "reward_min": min_rewards,
    "reward_max": max_rewards,
    "ep_length_mean": mean_lengths,
    "ep_length_std": std_lengths,
})

print(df)

    timesteps  reward_mean  reward_std  reward_min  reward_max  \
0       25000    -1.609093   45.790207  -56.521549   64.949600   
1       50000   127.493851  110.245331  -14.501663  284.612305   
2       75000   317.928802  166.067612   19.299316  487.246826   
3      100000   237.877808  157.017258  -38.870956  431.911713   
4      125000   240.043549  121.342567   -1.311722  317.853485   
5      150000   414.634125   68.650063  290.800110  497.168701   
6      175000   545.930298  209.050110  293.958008  773.584534   
7      200000   517.367798  226.963226   69.015427  681.653076   
8      225000   451.074768  168.130478  239.156464  720.431152   
9      250000   350.324646  298.184265  -15.376671  768.342896   
10     275000   347.667633  112.214233  208.991455  522.787598   
11     300000   379.203552  279.090881   37.325302  875.620056   
12     325000   650.030090  159.381912  375.405426  853.650696   
13     350000   513.882080  248.428726  222.976501  818.761780   
14     375